In [10]:
import pandas as pd
import numpy as np

from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from catboost import CatBoostClassifier

In [11]:
df = pd.read_csv('../data/processed/train.csv')

X = df.drop(columns='Survived')
y = df['Survived']

In [12]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

In [18]:
models = {
    'LogisticRegression' : LogisticRegression(max_iter=1000, solver='liblinear'),
    'KNN' : KNeighborsClassifier(n_neighbors=5),
    'DecisionTreeClassifier' : DecisionTreeClassifier(
        max_depth=5, 
        random_state=42
        ),
    'Random Forest' : RandomForestClassifier(
        n_estimators=200,
        max_depth=7,
        random_state=42
    ),
    'Catboost': CatBoostClassifier(
        iterations=300,
        depth=5,
        learning_rate=1e-2,
        loss_function='Logloss',
        verbose=0,
        random_state=42)
}


In [19]:
results = []

for name, model in models.items():
    scores = cross_val_score(
        model,
        X,
        y,
        cv=cv,
        scoring='accuracy'
    )

    results.append({
        'model': name,
        'mean accuracy': scores.mean(),
        'std accuracy' : scores.std(),
        'fold scores' : scores
    })

In [20]:
results_df = pd.DataFrame(results)
results_df = results_df.sort_values('mean accuracy', ascending=False)

results_df

,model,mean accuracy,std accuracy,fold scores
4,Catboost,0.830513,0.013177,"[0.8435754189944135, 0.8146067415730337, 0.825..."
3,Random Forest,0.826025,0.008206,"[0.8379888268156425, 0.8146067415730337, 0.831..."
0,LogisticRegression,0.808072,0.014049,"[0.8156424581005587, 0.8146067415730337, 0.786..."
2,DecisionTreeClassifier,0.801299,0.029890,"[0.8435754189944135, 0.7752808988764045, 0.797..."
1,KNN,0.621750,0.030948,"[0.6424581005586593, 0.6460674157303371, 0.567..."


Best results are:
    1. Catboost - 0.8305 mean accuracy but high std
    2. RF - 0.8260 mean accuracy but less std